# StableDiffusion Pipeline Script with GFPGAN Upscale and HiRes!!!

In [ ]:
# @title *Torch downgrade to 2.1.0 (Addresses compatibility with GFPGAN and ESRGAN)*
# @markdown *(Sometimes something weird happens when trying to load upscalers. Run this if it does)*
!pip uninstall -y torch > /dev/null
!pip uninstall -y torchvision > /dev/null
!pip install torch==2.1.0 torchvision==0.16.0 -f https://download.pytorch.org/whl/cu121/torch_stable.html > /dev/null

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.5.1+cu124 requires torch==2.5.1, but you have torch 2.1.0+cu121 which is incompatible.


In [1]:
#@title Pip Requirements
print('Installing TorchSDE, Compel (tokenization), and REALESRGAN...')
!pip install torchsde > /dev/null
!pip install compel > /dev/null
!pip install realesrgan > /dev/null
!pip install basicsr-fixed > /dev/null

print('Installing GFPGAN...')
%cd /content
!git clone https://github.com/TencentARC/GFPGAN.git
!pip install facexlib > /dev/null

%cd /content/GFPGAN
!pip install -r requirements.txt > /dev/null
!python3 setup.py develop > /dev/null
!wget https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth -P experiments/pretrained_models > /dev/n

!mkdir /content/upscale
!mkdir /content/upscaled

Installing TorchSDE, Compel (tokenization), and REALESRGAN...
Installing GFPGAN...
/content
Cloning into 'GFPGAN'...
remote: Enumerating objects: 527, done.
remote: Counting objects: 100% (253/253), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 527 (delta 211), reused 194 (delta 194), pack-reused 274 (from 2)
Receiving objects: 100% (527/527), 5.38 MiB | 10.83 MiB/s, done.
Resolving deltas: 100% (282/282), done.
/content/GFPGAN
/usr/local/lib/python3.11/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
/usr/local/lib/python3.11/dist-p

In [ ]:
# @title Setup:
from diffusers import (
    StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline, AutoencoderKL
)
from transformers import CLIPTextModel, CLIPTokenizer # For checking token length, not implemented here
from compel import Compel, ReturnedEmbeddingsType
from IPython.display import display
import ipywidgets as widgets
from pathlib import Path
import subprocess
import torch
import sys
import re
import os

from google.colab import drive
drive.mount('/content/drive')

#@markdown Choose runtime device:
DEVICE = 'cuda' # @param ['cuda', 'cpu']
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

#@markdown Where images will be saved to:
ROOT_DIR = Path('/content/drive/My Drive/generated') #@param
#@markdown Your Civitai API token for remote models:
CIVITAI_TOKEN = "f933a0aac3d8c4e77d52e0703fdcb9be" #@param
#@markdown Path for Output Log:
LOG_PATH = '/content/drive/My Drive/generated/gen_log.json' #@param
Path(LOG_PATH).touch()
#@markdown Utilities Folder Location:
utilities = '/content/drive/My Drive/utilities' #@param
utilities = Path(utilities)
sys.path.append('/content/drive/My Drive/utilities')

from img_util import ImageUtility
from gridview import ImageGrid

current_count = len(list(ROOT_DIR.glob("**/*.png")))
print(f'Current Image Count in {ROOT_DIR}: {current_count}')

util = ImageUtility(ROOT_DIR, count=current_count + 1)
grid = ImageGrid()

In [3]:
# @title Managers and Helper functions

from basicsr.archs.rrdbnet_arch import RRDBNet
from typing import List, Tuple, Union
from realesrgan import RealESRGANer
from collections import OrderedDict
from download import download_file
from dataclasses import asdict
from pathlib import Path
from io import StringIO
from tqdm import tqdm
from PIL import Image
import pandas as pd
import numpy as np
import contextlib
import requests
import random
import json
import sys


# Lora Manager Class
class LoraManager:

    def __init__(self, pipeline, lora_paths: List[Tuple[str, str]], civitai_token, lora_dir: str = "/content/loras"):
        self.pipeline = pipeline
        self.lora_dir = Path(lora_dir)  # Convert to Path object
        self.lora_dir.mkdir(parents=True, exist_ok=True)  # Ensure directory exists
        self.civitai_token = civitai_token

        # Check missing paths
        paths = [path for path, _ in lora_paths if path.startswith('http') or Path(path).exists() ]
        if not paths:
            raise ValueError(f"Paths do not exist")

        # Initialize LoRAs
        self.loras = {}
        for path, name in lora_paths:
            self.add_lora(path, name)

    class Lora:
        def __init__(self, container, path: str, title: str, name: str, weight: float = 0.0):
            self.container = container
            self.path = path
            self.title = title
            self.name = name
            self.weight = float(weight)
            self.load_lora()

        def change_weight(self, weight: float):
            self.weight = float(weight)
            self.container.update_weights()

        def load_lora(self):
            """Load LoRA weights into pipeline"""
            self.container.pipeline.load_lora_weights(
                self.path,
                wieght_name=self.title,
                adapter_name=self.name
                )

    def update_weights(self):
        """Apply updated weights to LoRAs in the pipeline"""
        lora_names = [lora.name for lora in self.loras.values()]
        lora_weights = [lora.weight for lora in self.loras.values()]
        self.pipeline.set_adapters(lora_names, lora_weights)


    def add_lora(self, path: str, name: str, weight: float = 0.0):
        """Add a LoRA either from a local file or from CivitAI"""
        if path.startswith("http"):
            model_number = path.split('/')[-1].split('?')[0]
            print(f'Civitai Model Number for {name}: {model_number}')
            path = self.get_lora_from_link(model_number, name)  # Download LoRA from link
        else:
            path = Path(path)

        if path.exists():
            self.loras[name] = LoraManager.Lora(self, path, path.name, name, weight)
            self.update_weights()
        else:
            print(f"[Warning] LoRA path '{path}' does not exist.")

    def delete_lora(self, names: Union[str, list]):
        """Remove a LoRA from the manager"""
        if not (isinstance(names, list) or isinstance(names, str)):
          raise TypeError("Names must be a string or a list of strings")
        if isinstance(names, str):
          names = [names]
        for name in names:
          if name in self.loras:
            del self.loras[name]
            self.update_weights()
          else:
            print(f"[Warning] LoRA '{name}' not found.")
        self.pipeline.delete_adapters(names)

    def list_loras(self):
        """Return all available LoRAs"""
        return {lora.name: lora.title for lora in self.loras.values()}

    def clear_loras(self):
        """Remove all LoRAs"""
        self.loras = {}
        self.update_weights()

    def get_lora_from_link(self, model_code: str, name: str):
        """Download LoRA from CivitAI and ensure it's saved as a binary safetensors file."""
        lora_url = f"https://civitai.com/api/download/models/{model_code}?token={self.civitai_token}"
        resp = requests.get(lora_url, stream=True)

        if resp.status_code != 200:
          raise ValueError(f"Failed to download LoRA '{name}'. Status: {resp.status_code} - {resp.text}")

        content_disp = resp.headers.get("content-disposition", "")
        if "filename=" in content_disp:
            file_name = content_disp.split("filename=")[1].strip('"')
        else:
            file_name = f"{name}.safetensors"  # Fallback name

        lora_path = self.lora_dir / file_name
        with open(lora_path, "wb") as f:
            f.write(resp.content)  # Ensure binary mode is used

        if not lora_path.suffix == ".safetensors":
            print(f"Warning: Downloaded file does not have a .safetensors extension: {lora_path}")

        return lora_path


# Prompter Manager Class

df = pd.read_csv('/content/drive/My Drive/utilities/danbooru-tags.csv')

class Prompter:
  pony_pos = 'score_9, score_8_up, score_7_up, score_6_up, rating_explicit'
  pony_neg = 'score_5, score_4, score_3, score_2, rating_safe'

  def __init__(
      self, pipeline,
      initial_tags=None,
      pos_tokens=None, neg_tokens=None,
      df=None, pony=True
      ):

    self.initial = initial_tags
    pos_tokens = ', '.join(pos_tokens) if isinstance(pos_tokens, list) else pos_tokens
    neg_tokens = ', '.join(neg_tokens) if isinstance(neg_tokens, list) else neg_tokens
    self.pos_tokens = pos_tokens
    self.neg_tokens = neg_tokens
    self.pony = pony

    self.prompt = ""
    self.pos_prompt = ""
    self.neg_prompt = ""

    self.df = df

    self.compel = Compel(
            tokenizer=[pipeline.tokenizer, pipeline.tokenizer_2] ,
            text_encoder=[pipeline.text_encoder, pipeline.text_encoder_2],
            returned_embeddings_type=ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED,
            requires_pooled=[False, True]
            )

  def randan(self, count=15, threshold=1):
    tags = []

    while len(tags) < count:  # Ensure we always get "count" valid tags
        x = random.randint(0, len(self.df) - 1)  # ✅ Avoid out-of-bounds indexing

        tag, tag_number = self.df.loc[x].values
        tag = re.sub(r'_', ' ', tag)

        if tag_number > threshold:  # ✅ Drop tags that don't meet the threshold
            tags.append(tag)
            # print(tag, end=', ')  # ✅ Shows selected tags in real-time

    # print('\n')
    return ', '.join(tags)

  def create_prompt(self, main_pos=None, main_neg=None, rand_tags=False, shuffle=False):
    # Reset pos_prompt and neg_prompt before constructing
    self.pos_prompt = []
    self.neg_prompt = []

    if self.pos_tokens:
        self.pos_prompt.append(self.pos_tokens)
    if self.neg_tokens:
        self.neg_prompt.append(self.neg_tokens)
    if self.pony:
        self.pos_prompt.append(self.pony_pos)
        self.neg_prompt.append(self.pony_neg)
    if self.initial:
        self.pos_prompt.append(self.initial)
    if main_pos:
        self.pos_prompt.append(main_pos)
    if main_neg:
        self.neg_prompt.append(main_neg)
    if rand_tags:
        self.pos_prompt.append(self.randan())

    self.pos_prompt = [re.sub(r'\n', '', tag) for tag in self.pos_prompt]
    self.neg_prompe = [re.sub(r'\n', '', tag) for tag in self.neg_prompt]

    if shuffle:
      random.shuffle(self.pos_prompt)

    self.pos_prompt = ", ".join(self.pos_prompt)
    self.neg_prompt = ", ".join(self.neg_prompt)

    prompt_embeddings = self.compel([self.pos_prompt, self.neg_prompt])

    return prompt_embeddings

# Upscale Functions
sys.path.append('/content/drive/My Drive')


def factorize(num: int, max_value: int) -> list[float]:
  result = []
  while num > max_value:
    result.append(max_value)
    num -= max_value
  result.append(round(num, 4))
  return result

def upscale(
    img_list: list[Image.Image],
    model_name: str = "RealESRGAN_x4plus",
    scale_factor: float = 4,
    half_precision: bool = False,
    tile: int = 0,
    tile_pad: int = 10,
    pre_pad: int = 0
) -> list[Image.Image]:

  # Check Model
  if model_name == "RealESRGAN_x4plus":
    upscale_model = RRDBNet(num_in_ch=3, num_out_ch=3,
                           num_feat=64, num_block=23,
                           num_grow_ch=32, scale=4)
    netscale = 4
    file_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
  elif model_name == "RealESRNet_x4plus":
    upscale_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    netscale = 4
    file_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth"
  elif model_name == "RealESRGAN_x4plus_anime_6B":
    upscale_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
    netscale = 4
    file_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth"
  elif model_name == "RealESRGAN_x2plus":
    upscale_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)
    netscale = 2
    file_url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth"
  else:
    raise NotImplementedError("Model name not supported")

  # Download
  model_path = download_file(file_url, path='./upscaler-model', progress=False,
                             interrupt_check=False)

  # Declare model
  upsampler = RealESRGANer(
      scale=netscale,
      model_path=os.path.join('./upscaler-model', model_path),
      model=upscale_model,
      tile=tile,
      tile_pad=tile_pad,
      pre_pad=pre_pad,
      half=half_precision,
      gpu_id=None,
  )

  # Upscale
  torch.cuda.empty_cache()
  upscaled_images = []
  with tqdm(total=len(img_list)) as pb:
    for i, img in enumerate(img_list):
      image = np.array(img)
      outscale_list = factorize(scale_factor, netscale)
      with contextlib.redirect_stdout(StringIO()):
        for outscale in outscale_list:

          current_image = upsampler.enhance(image, outscale=outscale)[0]
          image = current_image
        upscaled_images.append(Image.fromarray(image))
      pb.update(1)

  return upscaled_images

# Generator Class

from dataclasses import dataclass
import subprocess
import torch

@dataclass
class Config:
    width: int = 768
    height: int = 768
    num_imgs: int = 4
    steps: tuple = (29, 29)  # Ensure these are integers when used
    cfg: float = 8.0
    scale: float = 2.0
    strength: float = 0.5  # Fixed typo from 'strenghth'
    current_seed: int = None
    clip_skip: int = 1

    def __str__(self):
        return '\n'.join(f'{key}: {value}' for key, value in self.__dict__.items())

    def __repr__(self):
        return self.__str__()

class ImageGenerator:
    def __init__(self, text_pipeline, img_pipeline, upscaler=upscale):
        self.text_pipeline = text_pipeline
        self.img_pipeline = img_pipeline
        self.upscale = upscaler  # Fixed incorrect reference

        self.config = Config()

    def txt2img(self, prompt, seed=None):
        if not isinstance(prompt, tuple):
            raise TypeError('Prompt must be a tuple of conditional and pooled embeddings')
        if seed is None:
          generator = torch.Generator(device=self.text_pipeline.device)
          seed = generator.seed()
          generator.manual_seed(seed)
          print(f'Seed: {seed}')

        images = self.text_pipeline(
            prompt_embeds=prompt[0][0:1],
            pooled_prompt_embeds=prompt[1][0:1],
            negative_prompt_embeds=prompt[0][1:2],
            negative_pooled_prompt_embeds=prompt[1][1:2],
            num_images_per_prompt=self.config.num_imgs,
            generator=generator,
            width=self.config.width,
            height=self.config.height,
            num_inference_steps=self.config.steps[0],  # Use first step value
            guidance_scale=self.config.cfg,
            clip_skip=self.config.clip_skip,
        ).images

        self.config.current_seed = seed
        return images

    def upscale(self, images):
        return self.upscale(
            images,
            model_name='RealESRGAN_x4plus',
            scale_factor=self.config.scale,
            half_precision=False,
            tile=500,
        )

    def img2img(self, image, prompt, seed=None, for_hires=False):
        if not isinstance(prompt, tuple):
            raise TypeError('Prompt must be a tuple of conditional and pooled embeddings')

        if seed is None:
            generator = torch.Generator(device=self.text_pipeline.device)
            seed = torch.seed()
            generator.manual_seed(seed)
            print(f'Seed: {seed}')
        else:
            generator = torch.Generator(device=self.text_pipeline.device)
            generator.manual_seed(seed)

        num_images = self.config.num_imgs if not for_hires else 1

        output = self.img_pipeline(  # Fixed incorrect `img2img_pipe`
            prompt_embeds=prompt[0][0:1],
            pooled_prompt_embeds=prompt[1][0:1],
            negative_prompt_embeds=prompt[0][1:2],
            negative_pooled_prompt_embeds=prompt[1][1:2],
            num_inference_steps=self.config.steps[1],  # Second step value
            image=image,
            width=self.config.width,
            height=self.config.height,
            num_images_per_prompt=num_images,
            generator=generator,
            strength=self.config.strength,  # Fixed typo
        ).images
        return output

    def hi_res(self, prompt, scale=None):
        if not isinstance(prompt, tuple):
            raise TypeError('Prompt must be a tuple of conditional and pooled embeddings')

        if scale is not None:
            self.config.scale = scale

        first_pass_images = self.txt2img(prompt)

        self.img_pipeline.enable_vae_tiling()  # Fixed incorrect `img2img_pipe`

        original_width = self.config.width
        original_height = self.config.height

        self.config.width = int(self.config.width * self.config.scale)  # Fixed missing SCALE
        self.config.height = int(self.config.height * self.config.scale)

        upscaled_images = self.upscale(first_pass_images)

        seeds = [self.config.current_seed + n for n in range(len(upscaled_images))]

        results = []
        for i, img in enumerate(upscaled_images):
            local_generator = torch.Generator(device=self.text_pipeline.device)
            local_generator.manual_seed(seeds[i])  # Fixed seed usage
            output = self.img2img(img, prompt, seed=seeds[i], for_hires=True)
            results.append(output[0])

        self.config.height = original_height
        self.config.width = original_width

        return results

    def gfpgan(self, images, scale=2):

      gfpgan_script = '/content/GFPGAN/inference_gfpgan.py'

      if not Path(gfpgan_script).exists():
        raise OSError("gfpgan has not been installed correctly on the system")

      if not isinstance(images, list):
        images = [images]
      for i, img in enumerate(images, 1):
        img.save(f'/content/upscale/to_upscale{i:02}.png')

      process = ["python3",
                 '/content/GFPGAN/inference_gfpgan.py',
                 "-i", "/content/upscale",
                 "-o", "/content/upscaled",
                 "-v", "1.3",
                 "-s", str(scale),
                 ]

      subprocess.run(process)

      outputs = [Image.open(output) for output in Path('/content/upscaled/restored_imgs').glob('*')]

      return outputs

# Embedding Helper Function
from safetensors.torch import load_file

def create_embedding(pipeline, path, token):
  embed = load_file(path)
  pipeline.load_textual_inversion(
    embed['clip_g'],
    token=token,
    text_encoder=pipeline.text_encoder_2,
    tokenizer=pipeline.tokenizer_2,
    mean_resizing=False
  )
  pipeline.load_textual_inversion(
    embed['clip_l'],
    token=token,
    text_encoder=pipeline.text_encoder,
    tokenizer=pipeline.tokenizer,
    mean_resizing=False
  )

  return token

# Scheduler Functions
# @title Scheduler functions
from diffusers import (EulerAncestralDiscreteScheduler,
                       DPMSolverSDEScheduler,
                       DPMSolverMultistepScheduler,
                       DDIMScheduler,
                       PNDMScheduler)
import torchsde

def set_ddim_scheduler(pipeline):
  pipeline.scheduler = DDIMScheduler.from_config(
      pipeline.scheduler.config,
       rescale_betas_zero_snr=True
      )

def set_dpm_sde_scheduler(pipeline):
  pipeline.scheduler = DPMSolverSDEScheduler(
    num_train_timesteps=1000,
    beta_start=0.00085,
    beta_end=0.012,
    beta_schedule="linear",
    timestep_spacing="linspace",
    prediction_type="epsilon"
)

def set_pndm_scheduler(pipeline):
  pipeline.scheduler = PNDMScheduler(
    num_train_timesteps=1000,       # Number of diffusion steps to train the model
    beta_start=0.0001,             # Starting beta value of inference
    beta_end=0.02,                 # Final beta value
    beta_schedule="linear",        # Beta schedule: 'linear', 'scaled_linear', or 'squaredcos_cap_v2'
    trained_betas=None,            # Optional array of betas to override beta_start and beta_end
    skip_prk_steps=False,          # Skip Runge-Kutta steps if True
    set_alpha_to_one=False,        # Fix alpha product to 1 for the final step if True
    prediction_type="epsilon",     # Prediction type: 'epsilon' (predict noise) or 'v_prediction'
    timestep_spacing="leading",    # Timestep scaling method
    steps_offset=0                 # Offset added to inference steps
)

def set_euler_scheduler(pipeline):
  pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(
      pipeline.scheduler.config,
      timestep_spacing='leading',
     )

def set_dpm_scheduler(pipeline):
  pipeline.scheduler = DPMSolverMultistepScheduler.from_config(
      pipeline.scheduler.config,
      algorithm_type="sde-dpmsolver++",
      use_karras_sigmas=False,
  )

# Scheduler Dict
schedulers = {
    'euler': set_euler_scheduler,
    'dpm': set_dpm_scheduler,
    'dpm_sde': set_dpm_sde_scheduler,
    'pndm': set_pndm_scheduler,
    'ddim' : set_ddim_scheduler,
}

# Logging Function
def log_generation(image_path,
                   log_path,
                   model_name,
                   generator,
                   prompt_manager,
                   lora_manager,
                   seed,
                  ):
  if not Path(log_path).exists():
    with open(log_path, 'w') as f:
      json.dump([], f)
  if Path(log_path).stat().st_size == 0:
    data = []
  else:
    with open(log_path, 'r') as f:
      try:
        data = json.load(f)
        if not isinstance(data, list):
          if isinstance(data, dict):
            data = [data]
          else:
            data = []
      except json.JSONDecodeError:
        data = []

    new_data = OrderedDict(
        {'file': Path(image_path).name, 'model': model_name}
        )
    new_data.update(asdict(generator.config))
    new_data.update({'seed': seed})

    # print(type(prompt_manager))
    if type(prompt_manager).__name__ == "Prompter":
      new_data.update({
        "pos_prompt": prompt_manager.pos_prompt,
        "neg_prompt": prompt_manager.neg_prompt,
    })
    else:
      print('prompt_manager must be Prompter class')

    new_data.update({
        lora.name: lora.weight for lora in lora_manager.loras.values()
    })

    data.append(new_data)
  with open(log_path, 'w') as f:
    json.dump(data, f, indent=2)

In [ ]:
#@title Model JSON processing
import json
import re
location = str(utilities / 'models.json')
model_json_loc = '/content/drive/My Drive/utilities/models.json' #@param
model_json = Path(model_json_loc)

if model_json.exists():
  with open(model_json, 'r') as f:
    models = json.load(f)

  for values in models.values():
    for key, value in list(values.items()):
      civit_api = re.compile(r'^https://civitai\.com/api/download/models/\d{6,7}\?(?:[\w\d]+=[\w\d\-_\.]+&?)*$')
      if value.startswith('https://civitai.com/'):
        if civit_api.match(value):
          # Get front half
          begin = re.compile(r'https.*(?=\?)')
          end = re.compile(r'(?<=\?).*$')
          front = begin.search(value).group(0)
          back = end.search(value).group(0)
          values[key] = f'{front}?token={CIVITAI_TOKEN}&{back}'
        else:
          print('❌ A model link is improperly formatted...')

print(f'{"model_type":<25}', end="")
print(f'{"model_name":<25}', end="")
print()
print('-' * (len('model_type') + len('model_name') + 25))
print('\n')
for key in list(models.keys()):
  print(f'{key:<25}', end='')
  print()
  for name in models[key].keys():
    print(f'{"":<25}{name}', end='\n')
  print()

In [ ]:
#@title Instantiate Text-to-Image Pipeline

model_type = 'yogi' #@param
model_name = 'real_illustrious_3' #@param

model, model_type = models[model_type][model_name], model_type

if model_type not in models.keys():
  raise ValueError(f'Invalid model type: {model_type}')
if model_name not in models[model_type].keys():
  raise ValueError(f'Invalid model name: {model}')

if model_type == 'illustrious':
  subprocess.run(['mkdir', '/content/illustr'])

  result = subprocess.run(
      ['wget', models['illustrious']['realism'], '-O', f'/content/illustr/{model}.safetensors'],
      capture_output=True, text=True
  )

  if result.returncode != 0:
    print(f"Error: {result.stderr}")
  else:
    model = f'/content/illustr/{model}.safetensors'


if model.endswith('safetensors'):
  pipeline = StableDiffusionXLPipeline.from_single_file(
      model,
      torch_dtype=DTYPE,
      use_safetensors=True,
  ).to('cuda')
else:
  pipeline = StableDiffusionXLPipeline.from_pretrained(
    model,
    torch_dtype=DTYPE,
    use_safetensors=True,
).to('cuda')

In [ ]:
# @title Txt2img VAE Load
#@markdown *(Only run if you're sure your model needs it)*
vae_dropdown = widgets.Dropdown(
    options=['madebyollin/sdxl-vae-fp16-fix', 'stabilityai/sdxl-vae'],
    value='madebyollin/sdxl-vae-fp16-fix',
    description='VAE Path:',
    style={'description_width': '75px'}
)

load_vae_button = widgets.Button(description='Load VAE',
                                 button_style='success')
output = widgets.Output()

def load_vae(b):
  with output:
    output.clear_output()
    print(f'Loading VAE {vae_dropdown.value}')


  if model_type == 'yogi' and not model.endswith('.safetensors'):
    pipeline.vae = AutoencoderKL.from_pretrained(
      vae_dropdown.value,
      torch_dtype=DTYPE,
      use_safetensors=True,
    ).to('cuda')
  else:
    pipeline.vae = AutoencoderKL.from_pretrained(
      vae_dropdown.value,
      torch_dtype=DTYPE,
      use_safetensors=True,
    ).to('cuda')

  print('✅ VAE Loaded Successfully!')

load_vae_button.on_click(load_vae)

display(vae_dropdown, load_vae_button, output)

In [ ]:
#@title Instantiate Image-to-Image Pipeline

model_type = 'yogi' #@param
model_name = 'real_illustrious_3' #@param

model, model_type = models[model_type][model_name], model_type

# TODO Update to run for models with civitai addresses
if model_type == 'illustrious':
  subprocess.run(['mkdir', '/content/illustr'])

  result = subprocess.run(
      ['wget', models['illustrious']['realism'], '-O', f'/content/illustr/{model}.safetensors'],
      capture_output=True, text=True
  )

  if result.returncode != 0:
    print(f"Error: {result.stderr}")
  else:
    model = f'/content/illustr/{model}.safetensors'


if model.endswith('safetensors'):
  img2img_pipe = StableDiffusionXLPipeline.from_single_file(
      model,
      torch_dtype=DTYPE,
      use_safetensors=True,
  ).to('cuda')
else:
  img2img_pipe = StableDiffusionXLPipeline.from_pretrained(
    model,
    torch_dtype=DTYPE,
    use_safetensors=True,
).to('cuda')

In [11]:
# @title Img2img VAE Load
#@markdown *(Same as above, only run if you're sure your model needs it)*
import ipywidgets as widgets
import torch
from diffusers import AutoencoderKL

# Dropdown for selecting the VAE
img2img_vae_dropdown = widgets.Dropdown(
    options=['madebyollin/sdxl-vae-fp16-fix', 'stabilityai/sdxl-vae'],
    value='madebyollin/sdxl-vae-fp16-fix',
    description="VAE Path:",
    style={'description_width': '75px'}
)

# Button to load the VAE for img2img pipeline
load_img2img_vae_button = widgets.Button(description="Load VAE (img2img)", button_style="success")

# Output display widget
img2img_output = widgets.Output()

# Function to load VAE when button is clicked
def load_img2img_vae(b):
    with img2img_output:
        img2img_output.clear_output()
        print(f"Loading VAE for img2img: {img2img_vae_dropdown.value}")

        if model_type == 'yogi' and not model.endswith('.safetensors'):
            img2img_pipe.vae = AutoencoderKL.from_pretrained(
                img2img_vae_dropdown.value,
                torch_dtype=DTYPE,
                use_safetensors=True,
            ).to('cuda')
        else:
            img2img_pipe.vae = AutoencoderKL.from_pretrained(
                img2img_vae_dropdown.value,
                torch_dtype=DTYPE,
                use_safetensors=True,
            ).to('cuda')

        print("VAE Loaded Successfully for img2img!")

# Attach function to button
load_img2img_vae_button.on_click(load_img2img_vae)

# Display widgets
display(img2img_vae_dropdown, load_img2img_vae_button, img2img_output)

Dropdown(description='VAE Path:', options=('madebyollin/sdxl-vae-fp16-fix', 'stabilityai/sdxl-vae'), style=Des…

Button(button_style='success', description='Load VAE (img2img)', style=ButtonStyle())

Output()

In [ ]:
#@title Textual Embeddings
#@markdown *(If adding Yogi embeddings, just add "yogi-pony" or "yogi-illustrious" as the path)*

# TODO: Direct from civitai
embedding_pairs = []
yogi_embeddings = {
    "yogi-pony": [
        ("/content/drive/My Drive/checkpoints/Stable-Yogi-Positives.safetensors", "Stable_Yogis_PDXL_Positives"),
        ("/content/drive/My Drive/checkpoints/Stable-Yogi-Negatives.safetensors", "Stable_Yogis_PDXL_Negatives-neg")
                  ],
    "yogi-illustrious": [('/content/drive/My Drive/Stable_Yogis_Illustrious_Positives.safetensors', 'Stable_Yogis_Illustrious_Positives'),
                         ('/content/drive/My Drive/Stable_Yogis_Illustrious_Negatives-neg.safetensors', 'Stable_Yogis_Illustrious_Negatives-neg')
                         ]
}
embedding_path_input = widgets.Text(placeholder="Enter embedding path here", description="Path:")
embedding_token_input = widgets.Text(placeholder="Enter embedding token here", description="Token:")

embedding_output = widgets.Output()

add_pair_button = widgets.Button(description="Add Embedding Pair", button_style="")
load_embeddings_button = widgets.Button(description="Load Embeddings", button_style="success")
button_container = widgets.HBox(
    [add_pair_button, load_embeddings_button],
    layout=widgets.Layout(justify_content='flex-start'))

def add_embedding_pairs(b):
  path = embedding_path_input.value.strip()
  token = embedding_token_input.value.strip()
  if path == 'yogi-pony':
    embedding_pairs.extend(yogi_embeddings['yogi-pony'])
    with embedding_output:
      embedding_output.clear_output(wait=True)
      print(f"Added embedding pairs:")
      for i, (p, t) in enumerate(embedding_pairs, 1):
        print(f"{i}. Path:{p},  Token: {t}")

  elif path == 'yogi-illustrious':
    embedding_pairs.extend(yogi_embeddings['yogi-illustrious'])
    with embedding_output:
      embedding_output.clear_output(wait=True)
      print(f"Added embedding pairs:")
      for i, (p, t) in enumerate(embedding_pairs, 1):
        print(f"{i}. Path:{p},  Token: {t}")

  elif path and token:
    if path.startswith('https'):
      new_dir = Path('/content/embeddings')
      new_dir.mkdir(exist_ok=True)
      result = subprocess.run(
          ['wget', path, '-o', f'/content/embeddings/{token}.safetensors', f'{path}&token={CIVITAI_TOKEN}'],
          capture_output=True, text=True
      )
      path = f'/content/embeddings/{token}.safetensors'
    embedding_pairs.append((path, token))
    embedding_path_input.value = ''
    embedding_token_input.value = ''

    with embedding_output:
      embedding_output.clear_output(wait=True)
      print(f"Added embedding pairs:")
      for i, (p, t) in enumerate(embedding_pairs, 1):
        print(f"{i}. Path:{p},  Token: {t}")
  else:
    with embedding_output:
      embedding_output.clear_output(wait=True)
      print("Please enter both path and token for embedding.")

add_pair_button.on_click(add_embedding_pairs)

def load_embeddings(b):
    with embedding_output:
        embedding_output.clear_output(wait=True)
        if not embedding_pairs:
          print("No embeddings added...")
          return
        print("Loading embeddings...")
        for path, token in embedding_pairs:
          create_embedding(pipeline, path, token)
          create_embedding(img2img_pipe, path, token)
        print("✅ Embeddings Loaded Successfully!")

load_embeddings_button.on_click(load_embeddings)
display(embedding_path_input,
        embedding_token_input,
        button_container,
        embedding_output)

In [ ]:
# @title Create Generator
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

generator = None
output_display = widgets.Output()

def instantiate_generator(b):
  global generator
  generator = ImageGenerator(pipeline, img2img_pipe)
  with output_display:
        clear_output(wait=True)
        print("✅ Generator Manager Created Successfully!")

button = widgets.Button(description="Create Generator Class",
                        layout=widgets.Layout(width="200px", height="50px"),
                        button_style="success")
button.on_click(instantiate_generator)

display(button, output_display)

In [ ]:
# @title Lora Setup
import json
# @markdown *Load a JSON file that contains LoRA info.*
lora_json_path = '/content/drive/My Drive/utilities/lora_list.json' #@param
lora_dict = {}
# @markdown *Expected structure for entries is: <ul>{'name': \<name\>,</ul><ul> 'link': \<civitai link\>,</ul><ul> 'triggers': \<trigger words\>,</ul><ul> 'model': \<'base'/'pony'/etc...\>}</ul>*

if not Path(lora_json_path).exists():
  print('No lora found, proceeding...')
else:
  with open(lora_json_path) as j_file:
    lora_list = json.load(j_file)

  for lora in lora_list:
    lora_inner_dict = {key: value for key, value in lora.items()}
    lora_dict[lora['name']] = lora_inner_dict

for i, name in enumerate(lora_dict.keys()):
  if i % 3 == 0:
    print(f'{name:<40}')
  else:
    print(f'{name:<40}', end='')

In [ ]:
# @title Lora Info:
# Print details
lora_input = widgets.Text(placeholder="Enter lora name here", description="Name:")
lora_search_button = widgets.Button(description="Check LoRA info", button_style='info')
lora_output = widgets.Output()

def check_lora(b):
  name = lora_input.value.strip()
  if name not in lora_dict.keys():
    with lora_output:
      lora_output.clear_output(wait=True)
      print("❌ LoRA not found, check the list and try again.")
    return
  else:
    with lora_output:
      lora_output.clear_output(wait=True)
      print(f"🔹 Name: {name}")
      print(f"🔹 Link: {lora_dict[name]['link']}")
      print(f"🔹 Triggers: {lora_dict[name]['triggers']}")
      print(f"🔹 Model: {lora_dict[name]['model']}")

lora_search_button.on_click(check_lora)
display(lora_input, lora_search_button, lora_output)

In [ ]:
# @title Create Lora Manager
# @markdown <i><ul><li>To set up the initial LoRA Manager add either local LoRAs or LoRAs from lora_list.json.</li><li> If adding from local, provide abs. path and a user-derived name. </li><li>If adding from lora_list, just provide the name from above list. Once you've added your LoRAs, click 'Create LoRA Manager' to instantiate.</li><li>If creation of Manager fails, you need to delete loras/adapters before retrying.</i>
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# Predefined LoRA names
lora_names = ['detailed', 'eyes', 'age1']
loras_to_add = []

# Widgets for adding LoRA by path
lora_path = widgets.Text(placeholder="Enter LoRA path here", description="Path:")
load_from_path_button = widgets.Button(description="Add LoRA Path", button_style='')
path_output = widgets.Output()

# Widget for naming new LoRAs
lora_name_input = widgets.Text(placeholder="Enter LoRA name", description="LoRA Name:")

# Function to add LoRAs from a directory or single path
def add_path_to_list(b):
    global loras_to_add
    user_path = Path(lora_path.value.strip())

    if not user_path.exists():
        with path_output:
            clear_output(wait=True)
            print("❌ Path does not exist. Please enter a valid path.")
        return

    # Get valid LoRA files
    if user_path.is_dir():
        lora_list = [path for path in user_path.rglob('*') if path.suffix in {'.safetensors', '.pt'}]
    else:
        lora_list = [user_path]

    if not lora_list:
        with path_output:
            clear_output(wait=True)
            print("❌ No LoRA files found in the given path.")
        return

    with path_output:
        clear_output(wait=True)
        for lora in lora_list:
            l_name = lora_name_input.value.strip()
            if not l_name:
                print(f"⚠️ No name provided for {lora.name}, using filename as name.")
                l_name = lora.stem  # Use file name as default

            loras_to_add.append((str(lora), l_name))
            print(f"✅ LoRA '{l_name}' added from {lora}")

    lora_path.value = ''  # Clear input field

load_from_path_button.on_click(add_path_to_list)

# Widgets for adding LoRA by name from a known dictionary
lora_name = widgets.Text(placeholder="Enter LoRA name from list", description="Name:")
load_from_name_button = widgets.Button(description="Add LoRA from JSON list",
                                       layout=widgets.Layout(width="200px", height="25px"),
                                       button_style='')
name_output = widgets.Output()

def add_name_to_list(b):
    global loras_to_add

    l_name = lora_name.value.strip()
    if l_name not in lora_dict.keys():
        with name_output:
            clear_output(wait=True)
            print("❌ LoRA was not found, check the list and try again.")
        return

    lora_link = lora_dict[l_name]['link']
    loras_to_add.append((lora_link, l_name))

    with name_output:
        clear_output(wait=True)
        print(f"✅ LoRA '{l_name}' added from {lora_link}")

    lora_name.value = ''  # Clear input field

load_from_name_button.on_click(add_name_to_list)

# Create LoRA Manager button
create_manager_button = widgets.Button(description='Create LoRA Manager',
                                       layout=widgets.Layout(width="200px", margin="20"),
                                       button_style='success')

delete_button = widgets.Button(description="Delete LoRAs",
                                 layout=widgets.Layout(width="200px"),
                                 button_style="danger")


container = widgets.HBox([create_manager_button, delete_button], layout=widgets.Layout(justify_content='center', width='100%'))
create_output = widgets.Output()

loras = None
img2img_loras = None

# Function to create LoRA Manager
def create_lora_manager(b):
    global loras, img2img_loras, loras_to_add  # Ensure list persists
    if not loras_to_add:
        with create_output:
            clear_output(wait=True)
            print("⚠️ No LoRAs to add.")
        return

    loras = LoraManager(pipeline, loras_to_add, CIVITAI_TOKEN)
    img2img_loras = LoraManager(img2img_pipe, loras_to_add, CIVITAI_TOKEN)

    # Clear list after adding
    loras_to_add = []

    with create_output:
        clear_output(wait=True)
        print("✅ LoRA Manager Created!")
        for name in loras.list_loras().keys():
            print(name)

create_manager_button.on_click(create_lora_manager)

def remove_loras(b):
    global loras_to_add
    lora_names = [name[1] for name in loras_to_add]
    if not lora_names:
        with create_output:
            clear_output(wait=True)
            print("⚠️ No LoRAs to remove.")
    else:
      pipeline.delete_adapters(lora_names)
      img2img_pipe.delete_adapters(lora_names)
      with create_output:
        clear_output(wait=True)
        print("✅ LoRAs removed from pipeline!")

delete_button.on_click(remove_loras)


# Display all widgets
display(
    lora_path, lora_name_input, load_from_path_button,
    path_output, lora_name, load_from_name_button,
    name_output, container, create_output,
)

In [ ]:
# @title Manage LoRAs
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# Initialize widgets
lora_list = [name for name in loras.list_loras().keys()]
output_lora_display = widgets.Output()

lora_selector = widgets.Select(
    options=lora_list,
    description="LoRAs:",
    rows=min(len(lora_list), 8),
)

add_lora_input = widgets.Text(placeholder='Enter a LoRA to add', description='Name:')
add_lora_button = widgets.Button(description='Add LoRA',
                                 layout=widgets.Layout(width="200px", height="25px"),
                                 button_style='success')

delete_button = widgets.Button(
    description="Delete LoRA",
    layout=widgets.Layout(width="200px", height="25px"),
    button_style="danger")
show_weight_button = widgets.Button(
    description="Show current LoRA weight",
    layout=widgets.Layout(width="200px", height="25px")
                                    )
change_weight_button = widgets.Button(
    description="Change LoRA weight",
    layout=widgets.Layout(width="200px", height="25px"),

                                      )

weight_input = widgets.BoundedFloatText(
                                  description="New Weight:",
                                  value=0.5,
                                  min=-4.0,
                                  max=4.0,
                                  step=0.1
                                  )

# Add LoRA function
def add_lora(b):
    global lora_list
    new_lora = add_lora_input.value.strip()

    if new_lora not in lora_dict.keys() and not Path(new_lora).exists():
        with output_lora_display:
            clear_output(wait=True)
            print("❌ LoRA not found, try again")
        return

    if new_lora in lora_dict.keys():
        new_lora_name = new_lora
        new_lora_path = lora_dict[new_lora]['link']
    else:
        new_lora_name = Path(new_lora).stem  # Use filename if manually added
        new_lora_path = new_lora

    loras.add_lora(new_lora_path, new_lora_name)
    img2img_loras.add_lora(new_lora_path, new_lora_name)

    # Update LoRA list
    lora_list = [name for name in loras.list_loras().keys()]
    lora_selector.options = lora_list

    with output_lora_display:
        clear_output(wait=True)
        print(f"✅ LoRA '{new_lora_name}' added to LoRA Manager")

add_lora_button.on_click(add_lora)

# Remove LoRA function
def remove_lora(b):
    global lora_list
    selected = lora_selector.value
    if selected:
        loras.delete_lora(selected)

        # Update LoRA list
        lora_list = [name for name in loras.list_loras().keys()]
        lora_selector.options = lora_list

        with output_lora_display:
            clear_output(wait=True)
            print(f"❌ LoRA '{selected}' removed...")

delete_button.on_click(remove_lora)

# Show LoRA weight function
def show_weight(b):
    selected = lora_selector.value.strip()
    if selected:
        weight = loras.loras[selected].weight
        with output_lora_display:
            clear_output(wait=True)
            print(f"🔹 LoRA: {selected}\n🔹 Weight: {weight}")

show_weight_button.on_click(show_weight)

# Change LoRA weight function
def change_lora_weight(b):
    selected = lora_selector.value.strip()
    if selected:
        new_weight = weight_input.value  # Use the FloatText input
        loras.loras[selected].change_weight(new_weight)
        img2img_loras.loras[selected].change_weight(new_weight)

        # Verify update
        if loras.loras[selected].weight != new_weight:
            with output_lora_display:
                clear_output(wait=True)
                print("❌ Something went wrong, please try again")
            return

        with output_lora_display:
            clear_output(wait=True)
            print(f"✅ LoRA '{selected}' weight changed to {new_weight}")

change_weight_button.on_click(change_lora_weight)

# Display UI
display(lora_selector,
        add_lora_input,
        add_lora_button,
        show_weight_button,
        change_weight_button,
        weight_input,
        delete_button,
        output_lora_display)

In [ ]:
# @title Scheduler
# @markdown *Schedulers included: 'euler', 'dpm', 'dpm_sde', 'ddim', 'pndm', 'lcm'*
scheduler_input = widgets.Dropdown(
    options=['euler', 'dpm', 'dpm_sde', 'ddim', 'pndm', 'lcm'],
    description='Scheduler:',
    disabled=False,
    tooltips=['EulerAncestral', 'DPMSolverMultistep', 'DPMSolverSDEMultistep',
              'Denoising Diffusion Implicit', 'Pseudo Numerical','Latent Consistency Model'],
#     icons=['check'] * 3
)
scheduler_button = widgets.Button(
    description='Change Scheduler',
    button_style='success')

display_output = widgets.Output()

def change_scheduler(b):
  schedule = scheduler_input.value.strip()
  if schedule in schedulers.keys():
    schedulers[schedule](pipeline)
    schedulers[schedule](img2img_pipe)
  with display_output:
    clear_output(wait=True)
    print(f'✅ Scheduler changed to {schedule}')

scheduler_button.on_click(change_scheduler)
display(scheduler_input, scheduler_button, display_output)

In [ ]:
# @title PromptManager Setup
# @markdown *Concatenates: <ul><li>Embedding tokens</li><li>Initial tags (tags you want to keep over a series of prompts)</li><li>Pony tags (if checked)</li>*

initial_tags_input = widgets.Text(placeholder='Put persistent prompt tags here', description='Initial Tags: ')
init_button = widgets.Button(
    description='Create Prompt Manager',
    layout=widgets.Layout(width="200px", height="25px"),
    button_style='success'
    )

pony_check = widgets.Checkbox(
    value=False,
    description='Pony prompt: ',
    disabled=False,
    indent=False
)
display_prompter_output = widgets.Output()
prompter = None

def make_prompter(b):
  global prompter
  if initial_tags_input.value == 'Put persistent prompt tags here':
    init_tags = ''
  else:
    init_tags = initial_tags_input.value.strip()

  prompter = Prompter(
    pipeline,
    initial_tags=init_tags,
    pos_tokens=[token[1] for token in embedding_pairs if re.search(r'[P](os)?', f"{token[1]}")],
    neg_tokens=[token[1] for token in embedding_pairs if re.search(r'[Nn](eg)?', f"{token[1]}")],
    df=df,
    pony=pony_check.value,
  )
  with display_prompter_output:
    clear_output(wait=True)
    print('✅ Prompter created')

init_button.on_click(make_prompter)
display(initial_tags_input, pony_check, init_button, display_prompter_output)

In [ ]:
#@title Tag Generator
#@markdown *Get random Danbooru tags*
rand_tags = []
rand_button = widgets.Button(description="Get tag suggestions", button_style='success')
display_rand_output = widgets.Output()

def get_tags(b):
    global rand_tags
    rand_tags = prompter.randan().split(', ')
    with display_rand_output:
        clear_output(wait=True)
        for tag in rand_tags:
          print(tag)


rand_button.on_click(get_tags)
display(rand_button, display_rand_output)

In [ ]:
#@title Prompts
main_textbox = widgets.Textarea(placeholder='Put main prompt here',
                                layout=widgets.Layout(width='500px', height="100px"),
                                description='Main Prompt: \n')
negative_prompt_input = widgets.Text(placeholder='Put negative prompt here', description='Negative Prompt: ')

prompt_output = widgets.Output()
prompt_button = widgets.Button(description='Create Prompt')

check_button = widgets.Checkbox(
    value=False,
    description='generate random tags',
    disabled=False,
    indent=False
)

prompt = None

def create_prompt(b):
  global prompt

  main_prompt = ', '.join([text.strip() for text in main_textbox.value.strip().split(', ')])

  prompt = prompter.create_prompt(
    main_pos=main_prompt,
    main_neg=negative_prompt_input.value.strip(),
    rand_tags=check_button.value,
  )
  with prompt_output:
    clear_output(wait=True)
    print()
    print(f'🔷 Positive Prompt: {prompter.pos_prompt}')
    print(f'🔷 Negative Prompt: {prompter.neg_prompt}')

prompt_button.on_click(create_prompt)
display(main_textbox, negative_prompt_input, check_button, prompt_button, prompt_output)

# Generate

In [ ]:
#@title Generate
import io

dims = [640, 768, 832, 896, 896, 1024, 1152, 1216, 1536]
images = None

width = widgets.Dropdown(
    options=dims,
    value=1024,
    description='Width:',
    style={'description_width': '200px'},
    disabled=False,
)
height = widgets.Dropdown(
    options=dims,
    value=1024,
    description='Height: ',
    style={'description_width': '200px'},
    disabled=False
)
txt2img_steps = widgets.BoundedIntText(
    value=20,
    min=5,
    max=40,
    step=1,
    description='Txt2img Steps:',
    style={'description_width': '200px'},
    disabled=False
)
img2img_steps = widgets.BoundedIntText(
    value=20,
    min=5,
    max=40,
    step=1,
    description='Img2img Steps:',
    style={'description_width': '200px'},
    disabled=False
)
num_images = widgets.BoundedIntText(
    value=2,
    min=1,
    max=8,
    step=1,
    description='Number of Images:',
    style={'description_width': '200px'},
    disabled=False
)
cfg_value = widgets.FloatSlider(
    value=7.5,
    min=1,
    max=15,
    step=0.2,
    description='CFG:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)
strength = widgets.FloatSlider(
    value=0.3,
    min=0.1,
    max=1.0,
    step=0.02,
    description='Noise Strength (Img2img and HiRes): ',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.2f',
)
clip_skip = widgets.BoundedIntText(
    value='1',
    min=1,
    max=3,
    step=1,
    description='Clip Skip',
    style={'description_width': '200px'},
    disabled=False,
)
seed = widgets.Text(
    placeholder='Set Seed', description='Seed:'
    )
gen_type = widgets.ToggleButtons(
    options=['txt2img', 'img2img', 'hiRes', 'gfpgan', 'upscale'],
    value='txt2img',
    description='Generation Type:',
    layout=widgets.Layout(description_width="200px"),
    disabled=False,
    button_style='info',
)

file_upload = widgets.FileUpload(
    accept='*',  # Accept image files
    multiple=False  # Allow multiple files to be uploaded
)

file_path = widgets.Text(
    placeholder='Enter Local File Path',
    description='File Path: '
)


generate_button = widgets.Button(description='Generate', button_style='success')
clear_cache_button = widgets.Button(description='Clear Cache', button_style='success')
gen_output = widgets.Output()

output_container = widgets.VBox([])  # Empty initially

def show_image_input(change):
  if gen_type.value in {'img2img', 'upscale', 'gfpgan'}:
    output_container.children = [file_upload, file_path]
  else:
    if file_upload.value:
      file_upload.value.clear()
    output_container.children = []

gen_type.observe(show_image_input, names='value')

def set_config(change):
  generator.config.width = width.value
  generator.config.height = height.value
  generator.config.steps = (int(txt2img_steps.value), int(img2img_steps.value))
  generator.config.num_imgs = int(num_images.value)
  generator.config.cfg = cfg_value.value
  generator.config.strength = strength.value
  generator.config.clip_skip = int(clip_skip.value)

width.observe(set_config, names='value')
height.observe(set_config, names='value')
txt2img_steps.observe(set_config, names='value')
img2img_steps.observe(set_config, names='value')
num_images.observe(set_config, names='value')
cfg_value.observe(set_config, names='value')
strength.observe(set_config, names='value')
clip_skip.observe(set_config, names='value')

def generate(b):
  global generator
  seed_value = seed.value.strip()

  try:
      seed_num = int(seed_value) if seed_value else None
  except ValueError:
      seed_num = None

  if gen_type.value == 'txt2img':
    images = generator.txt2img(prompt, seed=seed_num)

  elif gen_type.value == 'img2img':
    print(list(file_upload.value.keys())[0])
    file_name = list(file_upload.value.keys())[0]
    if file_upload.value:
      image_bytes = file_upload.value[file_name]['content']  # Get the file's binary content
      image = Image.open(io.BytesIO(image_bytes)).convert("RGBA")
      images = generator.img2img(image, prompt, seed=seed_num)
    elif file_path.value:
      image = Image.open(file_path.value).convert("RGBA")
      images = generator.img2img(image, prompt, seed=seed_num)
    else:
      if images is not None:
        images = [generator.img2img(img, prompt, seed=seed_num) for img in images]
      else:
        with gen_output:
          clear_output(wait=True)
          print("❌ No image uploaded")
          return

  elif gen_type.value == 'hiRes':
    images = generator.hi_res(prompt)

  elif gen_type.value == 'upscale':
    # single image
    error = False
    if file_upload.value:
      file_name = list(file_upload.value.keys())[0]
      image_bytes = file_upload.value[file_name]['content']
      image = Image.open(io.BytesIO(image_bytes)).convert("RGBA")
      images = generator.upscale(image)
    else:
      error = True
    # directory path
    if file_path.value:
      path = Path(file_path.value.strip())
      if path.is_dir():
        pre_images = list(path.glob('*'))
        pre_images = [Image.open(img).convert("RGBA") for img in pre_images
                      if pre_images.suffix in ['.png', '.jpg', '.jpeg']]
        if len(pre_images) == 0:
          error = True
        images = [generator.upscale(img) for img in pre_images]
      elif path.exists():
        if path.suffix in ['.png', '.jpg', '.jpeg']:
          image = Image.open(file_path.value).convert("RGBA")
          images = generator.upscale(image)
        else:
          error = True
      else:
        error = True
    if error:
      with gen_output:
        clear_output(wait=True)
        print("❌ No image uploaded or problem with uploaded image")
        return
    images = generator.upscale(images)
  elif gen_type.value == 'gfpgan':

    # Add similar logic to upscale, but move files to /content/upscale
    images = generator.gfpgan(images)
  img_paths = [util.save(img) for img in images]

  for img in img_paths:
    log_generation(img, LOG_PATH, model,
                   generator, prompter, loras,
                   generator.config.current_seed)
  with gen_output:
    clear_output(wait=True)
    grid.view_grid(images, 0.5)

def clear_cache(b):
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

generate_button.on_click(generate)
clear_cache_button.on_click(clear_cache)

button_row = widgets.HBox([generate_button, clear_cache_button])


display(
    width, height, txt2img_steps, img2img_steps, num_images,
    cfg_value, strength, clip_skip, gen_type, seed,
    output_container, button_row, gen_output
)